# Preprocessing — King County House Sales Dataset

**Project:** Geospatial Valuation  
**Task:** Clean and preprocess the raw dataset based on issues flagged in `data_inspection.ipynb`  
**Dataset:** `kc_house_data.csv`  
**Owner:** Shais013

**Steps covered in this notebook:**
- Drop irrelevant identifier columns
- Parse and extract features from `date`
- Fix `sqft_basement` (text column with `?` placeholders)
- Handle missing values (`waterfront`, `view`, `yr_renovated`)
- Fix `yr_renovated` encoding + create `is_renovated` flag
- Fix bedroom outlier (33-bedroom entry)
- Remove price outliers (IQR method)
- Engineer `house_age` feature
- Save cleaned dataset

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded successfully.")

## 2. Load Raw Dataset

In [ ]:
from pathlib import Path

BASE_DIR = Path().resolve()
candidate_paths = [
    BASE_DIR / 'kc_house_data.csv',
    BASE_DIR.parent / 'Dataset' / 'kc_house_data.csv',
    BASE_DIR / 'Dataset' / 'kc_house_data.csv',
]

csv_path = next((p for p in candidate_paths if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError("kc_house_data.csv not found in any expected location")

df = pd.read_csv(csv_path)
print(f"Dataset loaded from: {csv_path}")
print(f"Raw shape: {df.shape}")
df.head()

## 3. Drop Identifier Column — `id`

`id` is a property/parcel identifier — it carries no predictive signal for price and would only add noise or cause leakage. Dropped immediately.

In [ ]:
df.drop(columns=['id'], inplace=True)
print(f"After dropping 'id': {df.shape}")
print("Remaining columns:", list(df.columns))

## 4. Parse `date` Column → Extract `sale_year` and `sale_month`

`date` is stored as a raw string (e.g. `10/13/2014`). We parse it and extract `sale_year` and `sale_month` as usable numeric features, then drop the original string column.

In [ ]:
df['date'] = pd.to_datetime(df['date'])
df['sale_year']  = df['date'].dt.year
df['sale_month'] = df['date'].dt.month
df.drop(columns=['date'], inplace=True)

print("Date parsed and extracted:")
print(f"  sale_year  unique values : {sorted(df['sale_year'].unique())}")
print(f"  sale_month unique values : {sorted(df['sale_month'].unique())}")

## 5. Fix `sqft_basement` — Object Column with `?` Placeholders

Flagged in inspection: `sqft_basement` is stored as text and contains `?` where the value is unknown. Steps:
1. Replace `?` with `NaN`
2. Convert to float
3. Fill `NaN` with `0` (unknown basement = assume no basement)

In [ ]:
question_marks = (df['sqft_basement'] == '?').sum()
print(f"'?' placeholders in sqft_basement: {question_marks}")

df['sqft_basement'] = df['sqft_basement'].replace('?', np.nan)
df['sqft_basement'] = pd.to_numeric(df['sqft_basement'])
df['sqft_basement'] = df['sqft_basement'].fillna(0)

print(f"sqft_basement dtype after fix: {df['sqft_basement'].dtype}")
print(f"Remaining nulls in sqft_basement: {df['sqft_basement'].isnull().sum()}")

## 6. Handle Missing Values — `waterfront` and `view`

Both flagged in inspection as having NaN values:
- `waterfront` (2376 nulls) — binary feature, fill with `0` (no waterfront)
- `view` (63 nulls) — ordinal score 0–4, fill with `0` (no view recorded)

In [ ]:
print("Before fill:")
print(f"  waterfront nulls : {df['waterfront'].isnull().sum()}")
print(f"  view nulls       : {df['view'].isnull().sum()}")

df['waterfront'] = df['waterfront'].fillna(0)
df['view']       = df['view'].fillna(0)

print("\nAfter fill:")
print(f"  waterfront nulls : {df['waterfront'].isnull().sum()}")
print(f"  view nulls       : {df['view'].isnull().sum()}")

## 7. Fix `yr_renovated` + Create `is_renovated` Flag

Flagged in inspection: `yr_renovated` uses `0` to mean "never renovated" and `NaN` for unknown — both map to the same meaning here. Steps:
1. Fill `NaN` with `0`
2. Create `is_renovated` binary flag (1 if renovated, 0 if not)
3. Keep `yr_renovated` as-is for potential use in `years_since_renovation`

In [ ]:
df['yr_renovated'] = df['yr_renovated'].fillna(0)
df['is_renovated'] = (df['yr_renovated'] > 0).astype(int)

print("yr_renovated nulls after fill:", df['yr_renovated'].isnull().sum())
print()
print("is_renovated value counts:")
print(df['is_renovated'].value_counts().rename({0: 'Not Renovated (0)', 1: 'Renovated (1)'}))

## 8. Engineer `house_age` Feature

A house's age at the time of sale is more informative than the raw `yr_built` value for modeling.  
`house_age = sale_year - yr_built`

In [ ]:
df['house_age'] = df['sale_year'] - df['yr_built']

print("house_age stats:")
print(df['house_age'].describe())

## 9. Fix Bedroom Outlier — 33 Bedrooms

Flagged in inspection: one row has 33 bedrooms with a low sqft_living — clearly a data entry error.  
We drop this row rather than capping, since it's a single corrupted record.

In [ ]:
print("Rows with bedrooms > 10:")
print(df[df['bedrooms'] > 10][['bedrooms', 'bathrooms', 'sqft_living', 'price']])

before = df.shape[0]
df = df[df['bedrooms'] <= 10]
after = df.shape[0]

print(f"\nDropped {before - after} row(s) with bedrooms > 10")
print(f"Shape after fix: {df.shape}")

## 10. Remove Price Outliers — IQR Method

Price is heavily right-skewed (max $7.7M vs median $450K). Extreme outliers can distort the model.  
We apply the standard IQR method: remove rows where price < Q1 - 1.5×IQR or > Q3 + 1.5×IQR.

In [ ]:
Q1  = df['price'].quantile(0.25)
Q3  = df['price'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(f"Price IQR bounds:")
print(f"  Q1     : ${Q1:,.0f}")
print(f"  Q3     : ${Q3:,.0f}")
print(f"  Lower  : ${lower:,.0f}")
print(f"  Upper  : ${upper:,.0f}")

before = df.shape[0]
df = df[(df['price'] >= lower) & (df['price'] <= upper)]
after = df.shape[0]

print(f"\nDropped {before - after} price outlier rows")
print(f"Shape after removal: {df.shape}")
print(f"Price range after   : ${df['price'].min():,.0f} — ${df['price'].max():,.0f}")

## 11. Final Null Check

In [ ]:
remaining_nulls = df.isnull().sum()
print("Remaining nulls per column:")
print(remaining_nulls[remaining_nulls > 0] if remaining_nulls.sum() > 0 else "None — dataset is fully clean.")
print(f"\nFinal dataset shape: {df.shape}")

## 12. Save Cleaned Dataset

In [ ]:
output_path = BASE_DIR.parent / 'Dataset' / 'kc_house_cleaned.csv'
df.to_csv(output_path, index=False)

print("Files saved:")
print(f"  kc_house_cleaned.csv — full cleaned dataset → {output_path}")

## 13. Preprocessing Summary

In [ ]:
print("=" * 60)
print("           PREPROCESSING SUMMARY")
print("=" * 60)
steps = [
    ("Dropped columns",            "id"),
    ("Parsed date",                "Extracted sale_year, sale_month"),
    ("Fixed sqft_basement",        "Replaced '?' with 0, converted to float"),
    ("Filled waterfront nulls",    "2376 NaN → 0"),
    ("Filled view nulls",          "63 NaN → 0"),
    ("Fixed yr_renovated",         "NaN → 0, created is_renovated flag"),
    ("Engineered house_age",       "sale_year - yr_built"),
    ("Fixed bedroom outlier",      "Dropped 33-bedroom row"),
    ("Removed price outliers",     "IQR method applied"),
    ("Saved cleaned dataset",      "kc_house_cleaned.csv"),
]
for k, v in steps:
    print(f"  {k:<30}: {v}")
print("=" * 60)
print(f"  Final shape: {df.shape}")
print("=" * 60)